In [5]:
import duckdb
from pathlib import Path
from IPython.display import display, Markdown

DB_PATH = "sead.duckdb"
DATA_DIR = Path("data/output")
YEAR = "2025"

CSV_OPTS = "header=true, nullstr=['', 'NULL', 'null', 'NA', 'N/A']"


def load_table(con: duckdb.DuckDBPyConnection,
             tabela: str,
             csv_path: Path,
             select_sql: str) -> None:
    """Cria tabela com tipos explícitos e carrega do CSV."""
    if not csv_path.exists():
        display(Markdown(f"[SKIP] {csv_path} não encontrado"))
        return
    con.execute(f"DROP TABLE IF EXISTS {tabela}")
    con.execute(f"""
        CREATE TABLE {tabela} AS
        {select_sql}
        FROM read_csv_auto('{csv_path.as_posix()}', {CSV_OPTS})
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    display(Markdown(f"[OK] {tabela:<15} {n:>10,} linhas"))

con = duckdb.connect(DB_PATH)

load_table(con, "atas",
    DATA_DIR  / f"atas_{YEAR}.csv",
    """
    SELECT
        CAST(numeroProcesso AS VARCHAR)   AS numeroProcesso,
        TRY_CAST(ano AS INTEGER)          AS ano,
        TRY_CAST(mes AS INTEGER)          AS mes,
        CAST(nomeOrgao AS VARCHAR)        AS nomeOrgao,
        CAST(tipoProcesso AS VARCHAR)     AS tipoProcesso,
        CAST(modalidade AS VARCHAR)       AS modalidade,
        CAST(situacao AS VARCHAR)         AS situacao,
        TRY_CAST(valorLicitado AS DOUBLE) AS valorLicitado
    """)

load_table(con, "contratacoes",
    DATA_DIR  / f"contratacoes_{YEAR}.csv",
    """
    SELECT
        CAST(numeroProcesso AS VARCHAR)    AS numeroProcesso,
        CAST(numeroLicitacao AS VARCHAR)   AS numeroLicitacao,
        CAST(cadastroCge AS VARCHAR)       AS cadastroCge,
        CAST(nomeOrgao AS VARCHAR)         AS nomeOrgao,
        CAST(modalidade AS VARCHAR)        AS modalidade,
        CAST(situacao AS VARCHAR)          AS situacao,
        TRY_CAST(valorEstimado AS DOUBLE)  AS valorEstimado,
        TRY_CAST(valorAdjudicado AS DOUBLE) AS valorAdjudicado
    """)

load_table(con, "contratos",
    DATA_DIR  / f"contratos_{YEAR}.csv",
    """
    SELECT
        CAST(numeroContrato AS VARCHAR)             AS numeroContrato,
        NULLIF(TRIM(CAST(numeroProcessoLicitacao AS VARCHAR)), '')
                                                    AS numeroProcessoLicitacao,
        CAST(registroCge AS VARCHAR)                AS registroCge,
        CAST(codigoOrgao AS VARCHAR)                AS codigoOrgao,
        NULLIF(TRIM(CAST(cnpjCpf AS VARCHAR)), '')  AS cnpjCpf,
        NULLIF(TRIM(CAST(cpfGestor AS VARCHAR)), '') AS cpfGestor,
        NULLIF(TRIM(CAST(matriculaGestor AS VARCHAR)), '')
                                                    AS matriculaGestor,
        CAST(nomeOrgao AS VARCHAR)                  AS nomeOrgao,
        CAST(nomeGestor AS VARCHAR)                 AS nomeGestor,
        TRY_CAST(valorTotal AS DOUBLE)              AS valorTotal,
        CAST(dataInicioVigencia  AS VARCHAR)        AS dataInicioVigencia,
        CAST(dataTerminoVigencia AS VARCHAR)        AS dataTerminoVigencia
    """)

load_table(con, "notas_empenho",
    DATA_DIR  / f"notas_empenho_{YEAR}.csv",
    """
    SELECT
        TRY_CAST(numeroEmpenho AS BIGINT)               AS numeroEmpenho,
        TRY_CAST(codigoOrgao  AS BIGINT)                AS codigoOrgao,
        TRY_CAST(ano          AS INTEGER)               AS ano,
        NULLIF(TRIM(CAST(contrato AS VARCHAR)), '')     AS contrato,
        NULLIF(TRIM(CAST(processo AS VARCHAR)), '')     AS processo,
        NULLIF(TRIM(CAST(numeroProcessoCompras AS VARCHAR)), '')
                                                        AS numeroProcessoCompras,
        TRY_CAST(codigoLicitacao AS DOUBLE)             AS codigoLicitacao,
        NULLIF(TRIM(CAST(registroCGE AS VARCHAR)), '')  AS registroCGE,
        NULLIF(TRIM(CAST(cpfCnpj AS VARCHAR)), '')      AS cpfCnpj,
        CAST(nomeOrgao AS VARCHAR)                      AS nomeOrgao,
        TRY_CAST(valorEmpenhado AS DOUBLE)              AS valorEmpenhado,
        TRY_CAST(valorLiquidado AS DOUBLE)              AS valorLiquidado,
        TRY_CAST(valorPago      AS DOUBLE)              AS valorPago
    """)

load_table(con, "liquidacoes",
    DATA_DIR  / f"liquidacoes_{YEAR}.csv",
    """
    SELECT
        TRY_CAST(numeroEmpenho AS BIGINT)                AS numeroEmpenho,
        TRY_CAST(codigoOrgao  AS BIGINT)                 AS codigoOrgao,
        TRY_CAST(anoExercicio AS INTEGER)                AS anoExercicio,
        NULLIF(TRIM(CAST(cnpjCpfCredor AS VARCHAR)), '') AS cnpjCpfCredor,
        TRY_CAST(numeroDocumento AS BIGINT)              AS numeroDocumento,
        CAST(nomeCredor AS VARCHAR)                      AS nomeCredor,
        TRY_CAST(valorEmpenhado AS DOUBLE)               AS valorEmpenhado,
        TRY_CAST(valorGD        AS DOUBLE)               AS valorGD
    """)

load_table(con, "servidores",
    DATA_DIR  / f"servidores_{YEAR}.csv",
    """
    SELECT
        NULLIF(TRIM(CAST(cpfServidor AS VARCHAR)), '')   AS cpfServidor,
        NULLIF(TRIM(CAST(matricula   AS VARCHAR)), '')   AS matricula,
        TRY_CAST(anoExercicio  AS INTEGER)               AS anoExercicio,
        TRY_CAST(mesReferencia AS INTEGER)               AS mesReferencia,
        TRY_CAST(periodo       AS INTEGER)               AS periodo,
        CAST(nomeServidor AS VARCHAR)                    AS nomeServidor,
        CAST(orgaoLotacao AS VARCHAR)                    AS orgaoLotacao,
        NULLIF(TRIM(CAST(cnpjOrgao AS VARCHAR)), '')     AS cnpjOrgao,
        CAST(nomeUnidadeTrabalho AS VARCHAR)             AS nomeUnidadeTrabalho,
        CAST(siglaPoder AS VARCHAR)                      AS siglaPoder,
        CAST(nomeCargo  AS VARCHAR)                      AS nomeCargo,
        CAST(tipoCargo  AS VARCHAR)                      AS tipoCargo,
        CAST(situacaoServidor AS VARCHAR)                AS situacaoServidor,
        CAST(dataAdmissao AS VARCHAR)                    AS dataAdmissao,
        TRY_CAST(valorBruto  AS DOUBLE)                  AS valorBruto,
        TRY_CAST(valorLiquido AS DOUBLE)                 AS valorLiquido
    """)

con.close()

[OK] atas                 2,500 linhas

[OK] contratacoes        46,848 linhas

[OK] contratos            7,480 linhas

[OK] notas_empenho      328,248 linhas

[OK] liquidacoes        381,176 linhas

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[OK] servidores       1,488,235 linhas

In [6]:
import duckdb
con = duckdb.connect("sead.duckdb")
con.execute(open("./lib/sead_indexes.sql").read())
con.execute(open("./lib/validation.sql").read())
con.execute(open("./lib/main_chain.sql").read())


In [ ]:

from IPython.display import display

df = con.execute("SELECT * FROM v_cadeia_principal LIMIT 10").df()

df2 = con.execute("""
    SELECT *
    FROM notas_empenho n
    JOIN liquidacoes l USING (numeroEmpenho, codigoOrgao)
    WHERE n.ano = l.anoExercicio
    LIMIT 10
""").df()

display(df2)
# Alertas do diagrama
print(con.execute("SELECT COUNT(*) FROM v_dup_atas").fetchone()[0], "processos duplicados em Atas")
print(con.execute("SELECT COUNT(*) FROM v_contratos_sem_processo").fetchone()[0], "contratos sem processo (esperado ~2.484)")

In [ ]:
import pandas as pd
atas = pd.read_csv("data/output/atas/atas_2025.csv")
contratos = pd.read_csv("data/output/contratos/contratos_2025.csv")
contratacoes = pd.read_csv("data/output/contratacoes/contratacoes_2025.csv")
liquidacoes = pd.read_csv("data/output/liquidacoes/liquidacoes_2025.csv")
notas_empenho = pd.read_csv("data/output/notas_empenho/notas_empenho_2025.csv")
servidores = pd.read_csv("data/output/servidores/servidores_2025.csv")


In [1]:
import duckdb
con = duckdb.connect("sead.duckdb")
con.close()